<a href="https://colab.research.google.com/github/Sakshi-2006/AIresumeAnalyser/blob/main/AIresumeAnalyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# AI RESUME ANALYZER
# =========================================================

# Install Required Libraries
!pip install transformers torch gradio pdfplumber python-docx sentence-transformers scikit-learn

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import gradio as gr
import pdfplumber
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# LOAD AI MODEL
# =========================================================

model = SentenceTransformer('all-MiniLM-L6-v2')

# =========================================================
# EXTRACT TEXT FROM PDF
# =========================================================

def extract_text_from_pdf(file_path):

    text = ""

    with pdfplumber.open(file_path) as pdf:

        for page in pdf.pages:

            extracted = page.extract_text()

            if extracted:
                text += extracted + "\n"

    return text

# =========================================================
# EXTRACT TEXT FROM DOCX
# =========================================================

def extract_text_from_docx(file_path):

    doc = Document(file_path)

    text = "\n".join(
        [para.text for para in doc.paragraphs]
    )

    return text

# =========================================================
# READ RESUME FILE
# =========================================================

def extract_resume_text(file):

    file_path = file.name

    if file_path.endswith(".pdf"):

        return extract_text_from_pdf(file_path)

    elif file_path.endswith(".docx"):

        return extract_text_from_docx(file_path)

    else:

        return "Unsupported file format"

# =========================================================
# SKILL EXTRACTION
# =========================================================

def extract_skills(text):

    skills_db = [

        # Programming
        "python",
        "java",
        "c++",
        "javascript",
        "html",
        "css",
        "react",
        "node.js",

        # AI/ML
        "machine learning",
        "deep learning",
        "tensorflow",
        "pytorch",
        "nlp",

        # Data
        "sql",
        "data analysis",

        # Cloud / DevOps
        "docker",
        "aws",
        "linux",
        "git",
        "github",

        # Soft Skills
        "communication",
        "leadership",
        "problem solving"
    ]

    text = text.lower()

    found_skills = []

    for skill in skills_db:

        if skill in text:

            found_skills.append(skill)

    return list(set(found_skills))

# =========================================================
# MAIN ANALYSIS FUNCTION
# =========================================================

def analyze_resume(resume_file, job_description):

    # Check input
    if resume_file is None:
        return "Please upload a resume."

    if job_description.strip() == "":
        return "Please paste a job description."

    # Extract Resume Text
    resume_text = extract_resume_text(resume_file)

    if resume_text == "Unsupported file format":
        return "Please upload PDF or DOCX file only."

    # =====================================================
    # AI SIMILARITY CHECK
    # =====================================================

    resume_embedding = model.encode([resume_text])

    jd_embedding = model.encode([job_description])

    similarity = cosine_similarity(
        resume_embedding,
        jd_embedding
    )[0][0]

    match_percentage = round(similarity * 100, 2)

    # =====================================================
    # SKILL MATCHING
    # =====================================================

    resume_skills = extract_skills(resume_text)

    jd_skills = extract_skills(job_description)

    matched_skills = list(
        set(resume_skills) & set(jd_skills)
    )

    missing_skills = list(
        set(jd_skills) - set(resume_skills)
    )

    # =====================================================
    # FINAL REPORT
    # =====================================================

    result = f"""

==================================================
                RESUME ANALYSIS REPORT
==================================================

📊 MATCH PERCENTAGE
--------------------------------------------------
{match_percentage} %

✅ MATCHING SKILLS
--------------------------------------------------
{', '.join(matched_skills) if matched_skills else 'No matching skills found'}

❌ MISSING SKILLS
--------------------------------------------------
{', '.join(missing_skills) if missing_skills else 'No missing skills'}

💡 SUGGESTIONS
--------------------------------------------------
"""

    if missing_skills:

        result += f"""

Add these skills to improve your resume:

{', '.join(missing_skills)}

Improve project descriptions and include
keywords related to the job description.
"""

    else:

        result += """

Excellent Resume!

Your resume aligns very well with
the job description.
"""

    return result

# =========================================================
# PROFESSIONAL GRADIO UI
# =========================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Resume Analyzer"
) as demo:

    gr.Markdown(
        """
        # AI Resume Analyzer

        Upload your resume and paste the job description.

        The AI will analyze:

        ✅ Resume Match Percentage
        ✅ Matching Skills
        ✅ Missing Skills
        ✅ Improvement Suggestions
        """
    )

    with gr.Row():

        # LEFT SIDE
        with gr.Column(scale=1):

            resume_input = gr.File(
                label="📄 Upload Resume (PDF/DOCX)"
            )

            jd_input = gr.Textbox(
                label="📝 Paste Job Description",
                placeholder="Paste complete job description here...",
                lines=20
            )

            analyze_button = gr.Button(
                "🚀 Analyze Resume",
                variant="primary"
            )

        # RIGHT SIDE
        with gr.Column(scale=1):

            output_box = gr.Textbox(
                label="📊 Resume Analysis Report",
                lines=30,
                max_lines=40,
                interactive=False,
                show_copy_button=True
            )

    # Button Click Action
    analyze_button.click(
        fn=analyze_resume,
        inputs=[
            resume_input,
            jd_input
        ],
        outputs=output_box
    )

# =========================================================
# LAUNCH APPLICATION
# =========================================================

demo.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 42.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2834/585694498.py:235: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94a19dbf311aab80fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
